# ARCH-LM Test in Python

A reusable implementation using `statsmodels`' `het_arch()` to test for ARCH effects (conditional heteroskedasticity / volatility clustering) in a time series or model residuals, with automatic decision-making.

**Key idea:** ARCH-LM tests whether the **squared** residuals are autocorrelated — i.e., whether volatility today depends on volatility in the past.

- **H0:** there are no ARCH effects (conditional variance is constant)
- **H1:** ARCH effects are present (conditional variance changes over time)

Unlike Ljung–Box on the raw residuals, a **small** ARCH-LM p-value is evidence of volatility clustering.

## The `arch_lm_test` Helper Function

This function runs the ARCH-LM test on a residual series and prints a full report, including the LM statistic, F statistic, and the decision.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.diagnostic import het_arch


def arch_lm_test(
    residuals,
    lags=10,
    significance=0.05,
    name="Residuals"
):
    """
    Perform ARCH-LM test and make a statistical decision.

    Parameters
    ----------
    residuals : array-like
        Residuals from a time-series model.
    lags : int
        Number of lags used in the ARCH-LM test.
    significance : float
        Significance level, e.g. 0.05.
    name : str
        Name of the residual series.
    """

    # Convert to pandas Series and remove missing values
    residuals = pd.Series(residuals).dropna()

    # Perform ARCH-LM test
    lm_stat, lm_pvalue, f_stat, f_pvalue = het_arch(
        residuals,
        nlags=lags
    )

    print("=" * 70)
    print(f"ARCH-LM TEST: {name}")
    print("=" * 70)

    print(f"Number of observations : {len(residuals)}")
    print(f"Number of lags        : {lags}")

    print("\nTest Results:")
    print(f"LM Statistic           : {lm_stat:.4f}")
    print(f"LM p-value             : {lm_pvalue:.4f}")
    print(f"F Statistic            : {f_stat:.4f}")
    print(f"F p-value              : {f_pvalue:.4f}")

    print(f"\nSignificance Level (\u03b1): {significance}")

    print("\nHypotheses:")
    print("H0: There are no ARCH effects.")
    print("H1: ARCH effects are present.")

    # Decision based on LM p-value
    print("\nDecision:")

    if lm_pvalue <= significance:
        print("Reject H0")
        print("Conclusion: ARCH effects are present.")
        print("Interpretation: Conditional heteroskedasticity")
        print("may be present in the residuals.")
        arch_effect = True

    else:
        print("Fail to Reject H0")
        print("Conclusion: No significant ARCH effects detected.")
        print("Interpretation: There is insufficient evidence")
        print("of conditional heteroskedasticity.")
        arch_effect = False

    print("=" * 70)

    return {
        "lm_statistic": lm_stat,
        "lm_pvalue": lm_pvalue,
        "f_statistic": f_stat,
        "f_pvalue": f_pvalue,
        "arch_effect": arch_effect
    }

## Example 1 — Test Random Noise

Generate independent random noise. Since the errors are i.i.d. with constant variance, we expect a **large** p-value and **fail to reject H0**.

In [ ]:
np.random.seed(42)

residuals = np.random.normal(
    loc=0,
    scale=1,
    size=1000
)

result = arch_lm_test(
    residuals,
    lags=10,
    significance=0.05,
    name="Random Noise"
)

Because the data are generated as independent random noise, you will generally get a relatively large p-value, e.g.:

```
ARCH-LM TEST: Random Noise
======================================================================
Number of observations : 1000
Number of lags         : 10

Test Results:
LM Statistic           : 8.23
LM p-value             : 0.6084
F Statistic            : 0.82
F p-value              : 0.6132

Significance Level (α): 0.05

Hypotheses:
H0: There are no ARCH effects.
H1: ARCH effects are present.

Decision:
Fail to Reject H0
Conclusion: No significant ARCH effects detected.
Interpretation: There is insufficient evidence
of conditional heteroskedasticity.
```

## Example 2 — Create Data With Volatility Clustering

To understand what ARCH-LM detects, generate an ARCH(1) process where today's variance depends directly on yesterday's squared shock:

$$\sigma_t^2 = \omega + \alpha\,\varepsilon_{t-1}^2$$

In [ ]:
np.random.seed(42)

n = 1000

# Generate ARCH(1) process
errors = np.zeros(n)
variance = np.zeros(n)

omega = 0.2
alpha = 0.8

random_shocks = np.random.normal(size=n)

variance[0] = omega / (1 - alpha)

for t in range(1, n):
    variance[t] = omega + alpha * errors[t-1]**2
    errors[t] = np.sqrt(variance[t]) * random_shocks[t]

result = arch_lm_test(
    errors,
    lags=10,
    significance=0.05,
    name="ARCH Process"
)

You should generally get a very small p-value, e.g. `LM p-value = 0.0000`.

Since p < 0.05, we **reject H0**.

**Conclusion:** ARCH effects are present. This indicates that the conditional variance changes over time.

## Example 3 — ARCH-LM on Stock Returns

For financial data, calculate returns first, then test them directly.

In [ ]:
df = pd.read_csv("stock_data.csv")

df["Return"] = df["Close"].pct_change()

returns = df["Return"].dropna()

result = arch_lm_test(
    returns,
    lags=10,
    significance=0.05,
    name="Stock Returns"
)

However, ARCH-LM is especially useful on **residuals** after modeling the conditional mean (e.g., an ARIMA fit) rather than on raw returns.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

model = ARIMA(returns, order=(1, 0, 0)).fit()
residuals = model.resid

result = arch_lm_test(
    residuals,
    lags=10,
    significance=0.05,
    name="Model Residuals"
)

## Decision Rule

The ARCH-LM test uses:

$$H_0: \text{No ARCH effects}$$
$$H_1: \text{ARCH effects are present}$$

At α = 0.05, the decision is:

| p-value | Decision | Conclusion |
|---|---|---|
| p ≤ 0.05 | Reject H0 | ARCH effects detected |
| p > 0.05 | Fail to reject H0 | No significant ARCH effects detected |

## Important Interpretation

```
p <= 0.05
    |
Reject H0
    |
ARCH effects detected
    |
Conditional variance is not constant
    |
Volatility clustering may exist
    |
Consider ARCH/GARCH-family modeling
```

Whereas:

```
p > 0.05
    |
Fail to Reject H0
    |
No significant ARCH effects detected
```

**One important point:** detecting ARCH effects does not automatically mean you must use GARCH. It means there is statistical evidence of conditional heteroskedasticity, after which you can investigate an appropriate volatility model and its specification.